In [ ]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Echo times
TE = np.arange(0, 302, 2)

# Parameter values
mwf_values = [5, 10, 15, 20, 25, 30, 35, 40, 45, 50]
t2mw_values = [10, 15, 20, 25, 30, 35, 40, 45, 50]
t2iew_values = [40, 60, 80, 100, 120, 140, 160, 180, 200]

# Default parameters
initial_mwf = 20
initial_t2mw = 20
initial_t2iew = 120

# Create figure
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=(
        "Biexponential T2 signal decay",
        "Relative myelin water contribution"
    )
)

# Store traces
traces = []

for mwf in mwf_values:
    for t2mw in t2mw_values:
        for t2iew in t2iew_values:

            A_mw = mwf / 100
            A_iew = 1 - A_mw

            S_mw = A_mw * np.exp(-TE / t2mw)
            S_iew = A_iew * np.exp(-TE / t2iew)

            S_total = S_mw + S_iew
            fraction_mw = 100 * S_mw / S_total

            visible = (
                mwf == initial_mwf
                and t2mw == initial_t2mw
                and t2iew == initial_t2iew
            )

            traces.extend([
                go.Scatter(
                    x=TE, y=S_mw,
                    name="Myelin water",
                    visible=visible
                ),
                go.Scatter(
                    x=TE, y=S_iew,
                    name="Intra- and extracellular water",
                    visible=visible
                ),
                go.Scatter(
                    x=TE, y=S_total,
                    name="Total signal",
                    visible=visible
                ),
                go.Scatter(
                    x=TE, y=fraction_mw,
                    name="MW contribution",
                    visible=visible
                )
            ])

# Add traces
for i, trace in enumerate(traces):
    if i % 4 == 3:
        fig.add_trace(trace, row=1, col=2)
    else:
        fig.add_trace(trace, row=1, col=1)

# Create slider steps
steps = []

for i, mwf in enumerate(mwf_values):
    for j, t2mw in enumerate(t2mw_values):
        for k, t2iew in enumerate(t2iew_values):

            index = (i * len(t2mw_values) * len(t2iew_values)
                     + j * len(t2iew_values) + k)

            visibility = [False] * len(traces)

            for n in range(4):
                visibility[4 * index + n] = True

            steps.append(
                dict(
                    method="update",
                    args=[
                        {"visible": visibility},
                        {"title": f"MWF = {mwf}%, T2 MW = {t2mw} ms, T2 IEW = {t2iew} ms"}
                    ],
                    label=f"{mwf}/{t2mw}/{t2iew}"
                )
            )

fig.update_layout(
    title="Myelin Water Fraction simulation",
    height=550,
    sliders=[
        dict(
            active=0,
            currentvalue={"prefix": "MWF / T2 MW / T2 IEW: "},
            steps=steps
        )
    ]
)

fig.update_xaxes(title_text="Echo time (ms)")
fig.update_yaxes(title_text="Normalized signal", row=1, col=1)
fig.update_yaxes(title_text="MW contribution (%)", row=1, col=2)

fig.show()